In [ ]:

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import sklearn
import dagshub
import mlflow
import xgboost
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression



c:\Users\Akash\OneDrive\Desktop\ml-ops-project\myenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
url = 'https://raw.githubusercontent.com/campusx-official/toy-datasets/main/student_performance.csv'
df = pd.read_csv(url)
df.head()

,IQ,CGPA,10th_Marks,12th_Marks,Communication_Skills,Placed
0,114,3.14,54,97,2.62,0
1,117,6.09,89,75,4.56,1
2,134,9.86,73,80,6.83,1
3,137,5.52,100,63,6.96,1
4,137,6.37,82,58,2.84,1


In [3]:
X = df.drop(columns=['Placed'])
y = df['Placed']

# data spliting
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(X,y,random_state= 42, test_size=0.2)


In [5]:
import dagshub
dagshub.init(repo_owner='akashkangule18', repo_name='ml-ops-project', mlflow=True)


Initialized MLflow to track repo "akashkangule18/ml-ops-project"

Repository akashkangule18/ml-ops-project initialized!

In [ ]:
mlflow.set_experiment("exp-2 with multiple baseline-models")

algorithms ={
    "LogisticRegression":LogisticRegression(),
    "XGBClassifier":XGBClassifier(),
    "RandomForestClassifier": RandomForestClassifier()
}

with mlflow.start_run(run_name='parent_run') as parent_run :
    for algo_name, algorithm in algorithms.items():
        with mlflow.start_run(run_name=algo_name, nested= True) as child_run:
            print(f"algorithm : {algo_name}")
            model = algorithm
            model.fit(X_train,y_train)

            y_pred = model.predict(X_test)

            from sklearn.metrics import accuracy_score, recall_score, precision_score, confusion_matrix

            acc = accuracy_score(y_test,y_pred)
            rcc = recall_score(y_test,y_pred)
            pcc = precision_score(y_test,y_pred)

        

            # saving file
            mlflow.log_artifact("exp-2.ipynb")

            # log params
            mlflow.log_param('test_size', 0.2)
            params = model.get_params()
            mlflow.log_params(params)

            # signature
            signature = mlflow.models.infer_signature(X_train, model.predict(X_train))

            # log model
            mlflow.sklearn.log_model(
                sk_model= model,
                name = algo_name,
                signature= signature
            )

            print(f"completed algorithm {algo_name}")
            print(f"accuracy_score {acc}")
            print(f"recall_score {rcc}")
            print(f"precision_score {pcc}")




2026/06/10 15:51:23 INFO mlflow.tracking.fluent: Experiment with name 'exp-2 with multiple baseline-models' does not exist. Creating a new experiment.


NameError: name 'LogisticRegression' is not defined